# Processed Data Quality Assessment

## Objective

Validate quality of **cleaned and processed datasets** after pipeline execution:
- Table inventory and row counts
- Join coverage across key relationships (FK integrity)
- Missing value patterns in processed tables
- Quality profile exports for audit trail

**Run this AFTER**: Quality checks on raw data (04_data_quality.ipynb)

**Output**: Reports in `reports/tables/` for downstream validation

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

DATA_PATH = '../../data/raw/'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Configuration & Data Loading

In [2]:
order_products_prior = pd.read_csv(f'{DATA_PATH}order_products__prior.csv')
order_products_train = pd.read_csv(f'{DATA_PATH}order_products__train.csv')
orders = pd.read_csv(f'{DATA_PATH}orders.csv')
products = pd.read_csv(f'{DATA_PATH}products.csv')
aisles = pd.read_csv(f'{DATA_PATH}aisles.csv')
departments = pd.read_csv(f'{DATA_PATH}departments.csv')

tables = {
    'orders': orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
    "products": products,
    "aisles": aisles,
    "departments": departments
}

## 1. Table Inventory

In [3]:
def build_table_inventory(table_map: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Generate table-level inventory: rows, columns, memory, duplicates."""
    rows = []
    for name, data in table_map.items():
        if data.empty:
            rows.append(
                {
                    "table": name,
                    "rows": 0,
                    "columns": 0,
                    "memory_mb": 0.0,
                    "duplicate_rows": 0,
                    "duplicate_pct": 0.0,
                    "status": "missing_or_empty",
                }
            )
            continue

        duplicate_rows = int(data.duplicated().sum())
        rows.append(
            {
                "table": name,
                "rows": int(len(data)),
                "columns": int(data.shape[1]),
                "memory_mb": round(float(data.memory_usage(deep=True).sum()) / (1024**2), 2),
                "duplicate_rows": duplicate_rows,
                "duplicate_pct": round(duplicate_rows / len(data) * 100, 3) if len(data) else 0.0,
                "status": "loaded",
            }
        )

    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)


inventory_df = build_table_inventory(tables)
print("📊 TABLE INVENTORY\n")
display(inventory_df)

📊 TABLE INVENTORY



,table,rows,columns,memory_mb,duplicate_rows,duplicate_pct,status
0,aisles,134,2,0.01,0,0.00,loaded
1,departments,21,2,0.00,0,0.00,loaded
2,order_products_prior,32434489,4,989.82,0,0.00,loaded
3,order_products_train,1384617,4,42.26,0,0.00,loaded
4,orders,3421083,7,332.71,0,0.00,loaded
5,products,49688,4,4.94,0,0.00,loaded


## 2. Join Coverage (Foreign Key Integrity)

In [4]:
def join_coverage(left: pd.DataFrame, right: pd.DataFrame, key: str, left_name: str, right_name: str) -> dict:
    """Check how many rows in left table match keys in right table (FK validation)."""
    if left.empty or right.empty:
        return {
            "left_table": left_name,
            "right_table": right_name,
            "join_key": key,
            "left_rows": len(left),
            "matched_rows": 0,
            "unmatched_rows": len(left),
            "match_rate_pct": 0.0,
            "note": "one_or_both_tables_missing",
        }

    right_keys = right[[key]].drop_duplicates()
    probe = left[[key]].merge(right_keys, on=key, how="left", indicator=True)
    matched_rows = int((probe["_merge"] == "both").sum())
    unmatched_rows = int((probe["_merge"] == "left_only").sum())

    return {
        "left_table": left_name,
        "right_table": right_name,
        "join_key": key,
        "left_rows": int(len(left)),
        "matched_rows": matched_rows,
        "unmatched_rows": unmatched_rows,
        "match_rate_pct": round(matched_rows / len(left) * 100, 3) if len(left) else 0.0,
        "note": "✅ ok" if unmatched_rows == 0 else "⚠️ ORPHAN RECORDS FOUND",
    }


join_quality_df = pd.DataFrame(
    [
        join_coverage(tables["order_products_prior"], tables["orders"], "order_id", "order_products_prior", "orders"),
        join_coverage(tables["order_products_train"], tables["orders"], "order_id", "order_products_train", "orders"),
        join_coverage(tables["order_products_prior"], tables["products"], "product_id", "order_products_prior", "products"),
        join_coverage(tables["order_products_train"], tables["products"], "product_id", "order_products_train", "products"),
        join_coverage(tables["products"], tables["aisles"], "aisle_id", "products", "aisles"),
        join_coverage(tables["products"], tables["departments"], "department_id", "products", "departments"),
    ]
)

print("\n🔗 JOIN COVERAGE CHECKS (Foreign Key Integrity)\n")
display(join_quality_df)


🔗 JOIN COVERAGE CHECKS (Foreign Key Integrity)



,left_table,right_table,join_key,left_rows,matched_rows,unmatched_rows,match_rate_pct,note
0,order_products_prior,orders,order_id,32434489,32434489,0,100.00,✅ ok
1,order_products_train,orders,order_id,1384617,1384617,0,100.00,✅ ok
2,order_products_prior,products,product_id,32434489,32434489,0,100.00,✅ ok
3,order_products_train,products,product_id,1384617,1384617,0,100.00,✅ ok
4,products,aisles,aisle_id,49688,49688,0,100.00,✅ ok
5,products,departments,department_id,49688,49688,0,100.00,✅ ok


## 3. Missing Values in Processed Tables

In [5]:
missing_by_column = (
    pd.concat(
        [
            pd.DataFrame({
                'table': name,
                'column': df.columns,
                'missing_count': df.isna().sum().values,
                'missing_pct': (df.isna().mean() * 100).values,
            })
            for name, df in tables.items()
        ],
        ignore_index=True,
    )
    .query('missing_count > 0')
    .sort_values(['missing_pct', 'missing_count'], ascending=False)
)

missing_by_column.head(20)

,table,column,missing_count,missing_pct
6,orders,days_since_prior_order,206209,6.03


In [6]:
# 1. สรุปภาพรวมจำนวนแถวและ Memory (แทน Inventory)
print("📊 --- Data Overview ---")
display(inventory_df[['table', 'rows', 'memory_mb']])

# 2. เช็คจุดที่มี Missing Values (ถ้ามี)
if not missing_by_column.empty:
    print("\n⚠️ --- Missing Values Detected ---")
    display(missing_by_column)
else:
    print("\n✅ No missing values found!")



📊 --- Data Overview ---


,table,rows,memory_mb
0,aisles,134,0.01
1,departments,21,0.00
2,order_products_prior,32434489,989.82
3,order_products_train,1384617,42.26
4,orders,3421083,332.71
5,products,49688,4.94



⚠️ --- Missing Values Detected ---


,table,column,missing_count,missing_pct
6,orders,days_since_prior_order,206209,6.03
